In [11]:
from pathlib import Path
from collections import Counter
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
FILE_PATH = Path(r"C:\Users\rando\Office Projects\rep_fetchapi\financial_headline_training_data.xlsx")
OUTPUT_PATH = FILE_PATH.with_name("financial_frequency_analysis.xlsx")

STOPWORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = re.sub(r"[^a-z\s]", " ", str(text).lower())
    return [
        w for w in text.split()
        if len(w) > 1 and w not in STOPWORDS
    ]

def ngrams(words, n):
    return [
        " ".join(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

overall = {
    "words": Counter(),
    "bigrams": Counter(),
    "trigrams": Counter()
}

categories = {}

xls = pd.ExcelFile(FILE_PATH)

for sheet in xls.sheet_names:

    df = pd.read_excel(FILE_PATH, sheet_name=sheet)

    if "Cleaned_Particulars" not in df.columns or "Category" not in df.columns:
        continue

    for _, row in df.iterrows():

        category = str(row["Category"]).strip()

        if pd.isna(row["Cleaned_Particulars"]):
            continue

        words = clean_text(row["Cleaned_Particulars"])

        if not words:
            continue

        data = {
            "words": words,
            "bigrams": ngrams(words, 2),
            "trigrams": ngrams(words, 3)
        }

        for key, values in data.items():
            overall[key].update(values)

        if category not in categories:
            categories[category] = {
                "words": Counter(),
                "bigrams": Counter(),
                "trigrams": Counter()
            }

        for key, values in data.items():
            categories[category][key].update(values)


with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:

    # Overall
    for key, counter in overall.items():

        pd.DataFrame(
            counter.most_common(),
            columns=[key[:-1], "frequency"]
        ).to_excel(
            writer,
            sheet_name=f"overall_{key}"[:31],
            index=False
        )

    # Category-wise
    for category, data in categories.items():

        safe_category = re.sub(r"[^A-Za-z0-9]", "_", category)

        for key, counter in data.items():

            pd.DataFrame(
                counter.most_common(),
                columns=[key[:-1], "frequency"]
            ).to_excel(
                writer,
                sheet_name=f"{safe_category}_{key}"[:31],
                index=False
            )

print(f"Done → {OUTPUT_PATH}")

Done → C:\Users\rando\Office Projects\rep_fetchapi\financial_frequency_analysis.xlsx


In [ ]:
FINANCIAL_PHRASES = [
    "non current",
    "current assets",
    "profit loss",
    "current liabilities",
    "increase decrease",
    "cash cash",
    "cash equivalents",
    "financial assets",
    "cash flow",
    "net cash",
    "financial liabilities",
    "comprehensive income",
    "equity share",
    "deferred tax",
    "share capital",
    "property plant",
    "operating activities",
    "plant equipment",
    "trade receivables",
    "investing activities",
    "financing activities",
    "trade payables",
    "current tax",
    "intangible assets",
    "short term",
    "equity liabilities",
    "income tax",
    "profit tax",
    "exceptional items",
    "long term",
    "working capital",
    "total equity",
    "items reclassified",
    "tax expense",
    "finance costs",
    "lease liabilities",
    "term borrowings",
    "cash flows",
    "loans advances",
    "face value",
    "net profit",
    "revenue operations",
    "total expenses",
    "tax liabilities",
    "capital changes",
    "work progress",
    "net increase",
    "total assets",
    "total income",
    "term loans",
    "extraordinary items",
    "operating profit",
    "employee benefits",
    "non controlling",
    "depreciation amortisation",
    "fixed assets",
    "bank balances",
    "continuing operations",
    "right use",
    "fair value",
    "total liabilities",
    "segment revenue",
]

In [5]:
import joblib

MODEL_PATH = "8_SEGMENT.pkl"

model = joblib.load(MODEL_PATH)

print("TYPE:", type(model))
print("CLASS:", model.__class__.__name__)

TYPE: <class 'sklearn.pipeline.Pipeline'>
CLASS: Pipeline


In [6]:
print("\nAttributes:")
print(model.__dict__.keys())


Attributes:
dict_keys(['steps', 'transform_input', 'memory', 'verbose'])


In [8]:
if hasattr(model, "steps"):
    for name, step in model.steps:
        print(name, "->", type(step))
        print(step.__dict__.keys())

tfidf -> <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
dict_keys(['input', 'encoding', 'decode_error', 'strip_accents', 'preprocessor', 'tokenizer', 'analyzer', 'lowercase', 'token_pattern', 'stop_words', 'max_df', 'min_df', 'max_features', 'ngram_range', 'vocabulary', 'binary', 'dtype', 'norm', 'use_idf', 'smooth_idf', 'sublinear_tf', '_tfidf', 'fixed_vocabulary_', '_stop_words_id', 'vocabulary_'])
classifier -> <class 'sklearn.linear_model._logistic.LogisticRegression'>
dict_keys(['penalty', 'C', 'l1_ratio', 'dual', 'tol', 'fit_intercept', 'intercept_scaling', 'class_weight', 'random_state', 'solver', 'max_iter', 'verbose', 'warm_start', 'n_jobs', 'n_features_in_', 'classes_', 'n_iter_', 'coef_', 'intercept_'])
